# RAG Eval — Шаг 1: Генерация тестового датасета Q&A

Для каждой заметки из `notes_dataset_v1.xlsx`:
1. Генерируем вопрос + эталонный ответ через `openai/gpt-5.1` (смарт-модель как оракул).
2. Определяем эталонный набор zettel-ов (ground truth) для метрик retrieval:
   через cosine-similarity эмбеддингов zettel-ов с текстом заметки.
3. Сохраняем в `synthetic_datasets/rag_qa_dataset.xlsx`.

**User в Neo4j:** `linker_eval_gpt-5.2`  
**Источник zettel-ов:** `synthetic_datasets/linker/reference_linker_gpt-5.2.xlsx`

In [1]:
import os
import sys
import uuid
import json
import ast
import time
import logging
from pathlib import Path
from datetime import datetime
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from zettelkasten.linker import LocalEmbeddingModel

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)
print('✅ imports ok')

/Users/nastya/github/Executive_Exocortex/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ imports ok


In [2]:
GENERATION_MODEL      = 'openai/gpt-5.1'
EVAL_USER_ID          = 'linker_eval_gpt-5.2'
MAX_WORKERS           = 15
GT_SIM_THRESHOLD      = 0.60    # порог similarity для zettel → note mapping

NOTES_PATH    = Path('synthetic_datasets/notes_dataset_v1.xlsx')
LINKER_PATH   = Path('synthetic_datasets/linker/reference_linker_gpt-5.2.xlsx')
OUTPUT_PATH   = Path('synthetic_datasets/rag_qa_dataset.xlsx')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'Paths configured. GT threshold: {GT_SIM_THRESHOLD}')

Paths configured. GT threshold: 0.6


In [3]:
df_notes  = pd.read_excel(NOTES_PATH)
df_linker = pd.read_excel(LINKER_PATH)

# фильтруем только карточки нашего пользователя
if 'user_id' in df_linker.columns:
    df_linker = df_linker[df_linker['user_id'] == EVAL_USER_ID].reset_index(drop=True)

print(f'Notes: {len(df_notes)} rows')
print(f'Linker zettels: {len(df_linker)} rows')
print(f'Linker cols: {list(df_linker.columns)}')
df_notes.head(3)

Notes: 200 rows
Linker zettels: 484 rows
Linker cols: ['Unnamed: 0', 'zettel_id', 'luhmann_id', 'content', 'thought_type', 'tags', 'is_root_topic', 'user_id', 'embedding', 'created_at', 'updated_at', 'similarity', 'action', 'reasoning', 'candidates_found', 'target_zettel_id']


,№,note_id,note_text,num_sentences,actual_sentences,domain,style,created_at
0,1,c09941ef-4576-4710-936d-dc12911f0ab7,Думаю по реструктуризации после разговора с Ма...,4,4,реструктуризация компании,рабочие мысли,2026-06-27 11:49:56
1,2,1d4f8aa3-9dc0-4898-b324-3b3c61a67436,Разобрал цифры по благотворительному проекту С...,3,3,благотворительный проект,анализ ситуации,2026-06-27 11:49:56
2,3,3a046e1a-56fb-4abb-afc4-cb65d85d3a24,Звонок с клиентом Северсталь Диджитал по проек...,6,6,работа с клиентами,записи после звонка,2026-06-27 11:49:58


In [4]:
embedder = LocalEmbeddingModel()

print('Embedding notes...')
note_texts = df_notes['note_text'].tolist()
note_embs  = np.array([embedder.embed_passage(t) for t in tqdm(note_texts, desc='notes')])
print(f'Note embeddings shape: {note_embs.shape}')

2026-06-28 17:12:55,916 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-28 17:12:55,917 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-28 17:12:56,009 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-base/d128750597153bb5987e10b1c3493a34e5a4502a/modules.json "HTTP/1.1 200 OK"
2026-06-28 17:12:56,203 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-06-28 17:12:56,204 | INFO | Loading SentenceTransformer model from intfloat/multilingual-e5-base.
2026-06-28 17:12:56,420 | INFO | HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
20

[EmbeddingModel] intfloat/multilingual-e5-base загружена за 7.7с (dim=768, device=mps)
Embedding notes...


notes: 100%|██████████| 200/200 [00:10<00:00, 18.46it/s]

Note embeddings shape: (200, 768)


In [5]:
def parse_embedding(val):
    """Парсим embedding из строки / списка / None."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    if isinstance(val, list):
        return np.array(val, dtype=np.float32)
    if isinstance(val, str):
        try:
            return np.array(ast.literal_eval(val), dtype=np.float32)
        except Exception:
            return None
    return None

raw_embs = [parse_embedding(e) for e in df_linker['embedding']]
valid_mask = [e is not None for e in raw_embs]
df_linker = df_linker[valid_mask].reset_index(drop=True)
zettel_embs = np.array([e for e in raw_embs if e is not None])

# Нормализуем zettel embeddings (на случай если они не нормализованы)
norms = np.linalg.norm(zettel_embs, axis=1, keepdims=True)
zettel_embs = zettel_embs / (norms + 1e-10)

print(f'Zettel embeddings shape: {zettel_embs.shape}')
print(f'Filtered zettels with valid embeddings: {len(df_linker)}')

Zettel embeddings shape: (484, 768)
Filtered zettels with valid embeddings: 484


In [6]:
# Матрица сходства: (n_zettels x n_notes)
sim_matrix = zettel_embs @ note_embs.T
print(f'Similarity matrix shape: {sim_matrix.shape}')

# Для каждого zettel → best matching note
best_note_idx = sim_matrix.argmax(axis=1)
best_note_sim = sim_matrix.max(axis=1)

df_linker['matched_note_id']  = None
df_linker['matched_note_sim'] = best_note_sim

for i in range(len(df_linker)):
    if best_note_sim[i] >= GT_SIM_THRESHOLD:
        df_linker.at[i, 'matched_note_id'] = df_notes.iloc[best_note_idx[i]]['note_id']

matched = df_linker['matched_note_id'].notna().sum()
print(f'Zettels matched to notes (sim >= {GT_SIM_THRESHOLD}): {matched} / {len(df_linker)}')

# note_id → список zettel_ids (ground truth для retrieval)
note_to_gt_zettels = defaultdict(list)
for _, row in df_linker.iterrows():
    if row['matched_note_id'] is not None:
        note_to_gt_zettels[row['matched_note_id']].append(row['zettel_id'])

unique_notes_with_gt = len(note_to_gt_zettels)
print(f'Notes with at least 1 matched zettel: {unique_notes_with_gt}')
print(f'Avg GT zettels per note: {np.mean([len(v) for v in note_to_gt_zettels.values()]):.1f}')

Similarity matrix shape: (484, 200)
Zettels matched to notes (sim >= 0.6): 484 / 484
Notes with at least 1 matched zettel: 131
Avg GT zettels per note: 3.7


In [7]:
QA_SYSTEM_PROMPT = """Ты — топ-менеджер, который пользуется персональной системой управления знаниями.
Тебе показывают одну из твоих старых заметок.

Твоя задача — придумать ОДИН реалистичный вопрос, который ты мог бы задать системе СПУСТЯ НЕКОТОРОЕ ВРЕМЯ
после того, как сделал эту заметку. Вопрос должен:
- Звучать как натуральный вопрос к своей базе знаний («напомни», «что мы решали», «кто отвечал», «какие цифры» и т.д.)
- Быть специфичным для данной заметки (содержать уникальные детали: имена, проекты, цифры)
- Иметь однозначный ответ в тексте заметки
- Не быть слишком простым («о чём эта заметка?») и не слишком широким

Также предоставь:
- reference_answer: точный и полный ответ на вопрос, основанный ТОЛЬКО на тексте заметки
- question_type: тип вопроса (factual / summary / action_items / risk / decision / detail)
- key_concepts: список 3-7 ключевых концептов из заметки, которые должны присутствовать в ответе"""


class QAPair(BaseModel):
    question: str = Field(description="Натуральный вопрос топ-менеджера к своей базе знаний")
    reference_answer: str = Field(description="Полный эталонный ответ на основе заметки")
    question_type: str = Field(description="Тип: factual / summary / action_items / risk / decision / detail")
    key_concepts: list[str] = Field(description="3-7 ключевых концептов для оценки покрытия ответа")


print('QA prompt schema ready')

QA prompt schema ready


In [8]:
save_lock = Lock()
results   = []


def generate_single_qa(row: dict) -> dict | None:
    note_id   = row['note_id']
    note_text = row['note_text']
    domain    = row.get('domain', '')
    style     = row.get('style', '')

    llm = ChatOpenAI(
        model=GENERATION_MODEL,
        temperature=0.7,
        api_key=os.getenv('LLM_API_KEY'),
        base_url=os.getenv('LLM_BASE_URL'),
    )

    try:
        qa: QAPair = llm.with_structured_output(QAPair).invoke([
            SystemMessage(content=QA_SYSTEM_PROMPT),
            HumanMessage(content=f"Заметка:\n{note_text}"),
        ])
        return {
            'question_id':             f'Q-{uuid.uuid4().hex[:8].upper()}',
            'source_note_id':          note_id,
            'note_text':               note_text,
            'note_domain':             domain,
            'note_style':              style,
            'question':                qa.question,
            'reference_answer':        qa.reference_answer,
            'question_type':           qa.question_type,
            'key_concepts':            json.dumps(qa.key_concepts, ensure_ascii=False),
            'ground_truth_zettel_ids': json.dumps(
                note_to_gt_zettels.get(note_id, []), ensure_ascii=False
            ),
            'n_gt_zettels':            len(note_to_gt_zettels.get(note_id, [])),
            'generated_at':            datetime.now().isoformat(),
            'model_name':              GENERATION_MODEL,
        }
    except Exception as e:
        logger.error(f'Error generating QA for note {note_id}: {e}')
        return None


rows = df_notes.to_dict('records')

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(generate_single_qa, r): r['note_id'] for r in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Generating Q&A'):
        res = future.result()
        if res is not None:
            with save_lock:
                results.append(res)

df_qa = pd.DataFrame(results)
df_qa['№'] = range(1, len(df_qa) + 1)
print(f'Generated: {len(df_qa)} Q&A pairs')
df_qa.head(3)

Generating Q&A:   0%|          | 0/200 [00:00<?, ?it/s]2026-06-28 17:13:16,608 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-06-28 17:13:16,610 | INFO | Retrying request to /chat/completions in 0.421826 seconds
2026-06-28 17:13:16,619 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-06-28 17:13:16,620 | INFO | Retrying request to /chat/completions in 0.464909 seconds
2026-06-28 17:13:17,205 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-06-28 17:13:17,205 | INFO | Retrying request to /chat/completions in 0.840266 seconds
2026-06-28 17:13:17,372 | INFO | HTTP Request: POST https://litellm.tokengate.ru/v1/chat/completions "HTTP/1.1 503 Service Unavailable"
2026-06-28 17:13:17,373 | INFO | Retrying request to /chat/completions in 0.909840 seconds
2026-06-28 17:13:18,281 | INFO | HTTP R

Generated: 199 Q&A pairs


,question_id,source_note_id,note_text,note_domain,note_style,question,reference_answer,question_type,key_concepts,ground_truth_zettel_ids,n_gt_zettels,generated_at,model_name,№
0,Q-0A1D247E,33ba4155-d599-4f78-b9f2-f18f10c6bd4a,Звонок с Андреем и Michael из DHL по поставкам...,операционная деятельность,записи после звонка,"Напомни, какой именно лимит на экспресс-достав...",Решили временно поднять лимит на экспресс до 1...,factual,"[""лимит на экспресс"", ""1,8 млн рублей"", ""проек...","[""fa44b3b5-f93c-40ee-bdee-870f0f1a08a1""]",1,2026-06-28T17:13:18.289649,openai/gpt-5.1,1
1,Q-FB4EB54F,142f182e-072c-4364-9058-c71303b5a194,Только что посмотрел с Олей и Mark дашборд в T...,аналитика и метрики,голосовая заметка,"Напомни, какую оценку по потере MRR называла К...",Катя оценивает потенциальное влияние на MRR ка...,factual,"[""Катя"", ""падение конверсии trial→оплата у Сбе...","[""cd4790f7-8b58-4c57-bfc3-13b1d2d3a464"", ""ec29...",5,2026-06-28T17:13:18.299467,openai/gpt-5.1,2
2,Q-8F323939,9c5f97c0-f0f3-4cc0-b272-5ddf4bab53b4,"Думаю по людям в команде Phoenix, нагрузка уже...",управление персоналом,рабочие мысли,"Напомни, к какому сроку мы рисковали сдвинуть ...",Если до 15 июля не закрыть двух backend-разраб...,factual,"[""проект Vega для MTS"", ""срок до 15 июля"", ""дв...",[],0,2026-06-28T17:13:18.446104,openai/gpt-5.1,3


In [11]:
df_qa.to_excel(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')

print('\n=== Dataset Statistics ===')
print(f'Total Q&A pairs:              {len(df_qa)}')
print(f'With GT zettels:              {(df_qa["n_gt_zettels"] > 0).sum()}')
print(f'Avg GT zettels per question:  {df_qa["n_gt_zettels"].mean():.2f}')
print(f'Questions with 0 GT zettels:  {(df_qa["n_gt_zettels"] == 0).sum()}')

print('\nQuestion type distribution:')
print(df_qa['question_type'].value_counts())

# Предупреждение: вопросы без GT zettel-ов будут пропущены в retrieval eval
zero_gt = (df_qa['n_gt_zettels'] == 0).sum()
if zero_gt > 0:
    print(f'\n⚠️  {zero_gt} questions have 0 GT zettels (threshold={GT_SIM_THRESHOLD}).')
    print('Consider lowering GT_SIM_THRESHOLD if too many are missing.')

Saved to synthetic_datasets/rag_qa_dataset.xlsx

=== Dataset Statistics ===
Total Q&A pairs:              199
With GT zettels:              131
Avg GT zettels per question:  2.43
Questions with 0 GT zettels:  68

Question type distribution:
question_type
factual         154
decision         32
action_items      6
risk              4
detail            2
summary           1
Name: count, dtype: int64

⚠️  68 questions have 0 GT zettels (threshold=0.6).
Consider lowering GT_SIM_THRESHOLD if too many are missing.


In [12]:
# Просмотр нескольких примеров
for i, row in df_qa.head(2).iterrows():
    print(f"{'='*80}")
    print(f"[{row['question_type'].upper()}] {row['question']}")
    print(f"\nAnswer: {row['reference_answer'][:300]}...")
    print(f"GT zettels ({row['n_gt_zettels']}): {row['ground_truth_zettel_ids'][:100]}...")

[FACTUAL] Напомни, какой именно лимит на экспресс-доставку мы тогда решили временно поднять в рамках проекта «Север» для клиента ВекторФарм из‑за задержки партии в Риге?

Answer: Решили временно поднять лимит на экспресс до 1,8 млн рублей....
GT zettels (1): ["fa44b3b5-f93c-40ee-bdee-870f0f1a08a1"]...
[FACTUAL] Напомни, какую оценку по потере MRR называла Катя из‑за падения конверсии trial→оплата у СберМаркета в проекте Orion, если тренд не развернуть?

Answer: Катя оценивает потенциальное влияние на MRR как минус 4,5 млн рублей, если тренд падения конверсии не развернуть....
GT zettels (5): ["cd4790f7-8b58-4c57-bfc3-13b1d2d3a464", "ec29692a-3b9b-46af-bc64-7b49e192bdfa", "3a47e2ef-cffb-4835...
